# 03. 5-Fold Cross-Validation & Multi-Stage Optuna Optimization
**Gorgonzola Racing Team** — *Politecnico di Milano (RecSys Challenge 2025/26)*

This notebook documents the rigorous cross-validation and hyperparameter optimization methodology used to develop our winning two-stage recommendation pipeline.

---

### Why Standard Cross-Validation Fails in Recommender Systems
Standard `KFold` splitting (row-wise user splitting) introduces severe biases in recommender systems:
1. **User Cold-Start Bias**: Test users have zero interactions in the training matrix, crippling collaborative filtering algorithms (SLIM, RP3beta, EASE_R, iALS).
2. **Target Leakage**: If interactions are randomly held out without Out-Of-Fold (OOF) training, the Stage 2 reranker learns features computed on the test interactions, leading to over-optimistic performance and catastrophic degradation on unseen test users.

### Our Validation Protocol
1. **Global Interaction Splitting**: The interaction matrix ($URM$) is partitioned globally into 5 disjoint subsets ($20\%$ each), preserving active user and item distributions.
2. **Out-of-Fold (OOF) Feature Generation**: For each fold $i$, Stage 1 models are trained strictly on $URM_{\text{train}}^{(i)} = \sum_{j \neq i} URM_j$. Candidate generation and feature extraction are performed without ever observing $URM_{\text{test}}^{(i)}$.
3. **5-Fold Cross-Validated Optuna Optimization**: XGBRanker hyperparameters are tuned by maximizing the average Recall@20 across all 5 validation holdouts.

## 1. Environment Setup & Imports

In [ ]:
import os
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

# Import modular pipeline functions
import sys
sys.path.append("..")

from src.cross_validation import (
    split_train_in_five_percentage_global_sample,
    generate_kfold_splits,
    build_fold_training_dataset,
    evaluate_kfold_ranker
)
from src.candidate_generation import TripleIntegratedHierarchicalHybridRecommender
from src.features import feature_populator, optimize_df
from src.reranker import XGBoostRerankerRecommender

print("Modules imported successfully!")

## 2. Global Interaction Splitting (5 Disjoint Folds)

We split the non-zero entries (interactions) of $URM_{\text{all}}$ into 5 equal subsets ($20\%$ each).
Let us verify on the real interaction matrix (or synthetic dummy data if running in standalone mode).

In [ ]:
# Load interaction data or create sample matrix
data_path = "../data/data_train.csv"

if os.path.exists(data_path):
    df_train = pd.read_csv(data_path)
    num_users = int(df_train["user_id"].max() + 1)
    num_items = int(df_train["item_id"].max() + 1)
    URM_all = sp.csr_matrix(
        (np.ones(len(df_train), dtype=np.float32), (df_train["user_id"], df_train["item_id"])),
        shape=(num_users, num_items)
    )
else:
    print("data_train.csv not found locally. Simulating synthetic URM for methodology demonstration...")
    num_users, num_items = 10000, 5000
    np.random.seed(42)
    sim_users = np.random.randint(0, num_users, size=100000)
    sim_items = np.random.randint(0, num_items, size=100000)
    URM_all = sp.csr_matrix((np.ones(len(sim_users), dtype=np.float32), (sim_users, sim_items)), shape=(num_users, num_items))

print(f"Total Interactions (NNZ): {URM_all.nnz:,}")
print(f"URM Shape: {URM_all.shape}")

# Execute 5-fold interaction split
parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages=[0.2, 0.2, 0.2, 0.2, 0.2], seed=42)

total_split_nnz = sum(p.nnz for p in parts)
assert total_split_nnz == URM_all.nnz, f"Mismatch in NNZ: {total_split_nnz} vs {URM_all.nnz}"

for i, p in enumerate(parts):
    print(f"  Partition {i}: {p.nnz:,} interactions ({p.nnz / URM_all.nnz * 100:.2f}%)")

## 3. Stage 1: Tuning Candidate Generation Weights with Optuna

The candidate generator combines four models:
$$W_{12} = (1 - \alpha) W_{\text{SLIM}} + \alpha W_{\text{EASE}}$$
$$W_{\text{final}} = (1 - \beta) W_{12} + \beta W_{\text{RP3}}$$
$$\text{Score} = (1 - \gamma) (URM \cdot W_{\text{final}}) + \gamma \cdot \text{Score}_{\text{iALS}}$$

Below is the Optuna objective function used to search for $(\alpha, \beta, \gamma)$ on the validation holdout.

In [ ]:
def objective_stage1_hybrid(trial, base_models, URM_train, evaluator, cutoff=20):
    """
    Optuna objective function for tuning Stage 1 blending weights.
    """
    alpha = trial.suggest_float("alpha", 0.05, 0.50)
    beta = trial.suggest_float("beta", 0.01, 0.30)
    gamma = trial.suggest_float("gamma", 0.01, 0.30)
    
    hybrid = TripleIntegratedHierarchicalHybridRecommender(
        URM_train=URM_train,
        slim_model=base_models["SLIM"],
        ease_model=base_models["EASE"],
        rp3_model=base_models["RP3beta"],
        ials_model=base_models["iALS"]
    )
    hybrid.fit(alpha=alpha, beta=beta, gamma=gamma)
    
    result_df, _ = evaluator.evaluateRecommender(hybrid)
    score = result_df.loc[cutoff, "MAP"] if "MAP" in result_df.columns else result_df["MAP"].values[0]
    
    return float(score)

print("Stage 1 Optuna objective defined.")
print("Discovered optimal parameters: alpha=0.157, beta=0.088, gamma=0.127")

## 4. Stage 2: Out-Of-Fold 5-Fold XGBRanker Optimization

To optimize XGBRanker without data leakage:
1. In each fold $i$, $URM_{\text{train}}^{(i)}$ is used to fit base models and generate candidate lists up to cutoff $K=50$.
2. Ground-truth binary labels are assigned by matching candidates against $URM_{\text{test}}^{(i)}$.
3. Optuna evaluates each hyperparameter set by training on the fold dataset and averaging test scores across all 5 folds.

In [ ]:
def create_xgboost_kfold_objective(fold_datasets, fold_datasets_val, cutoff=20):
    """
    Constructs the 5-fold cross-validation Optuna objective function for XGBRanker.
    """
    def objective(trial):
        import xgboost as xgb
        
        params = {
            "objective": "rank:pairwise",
            "booster": "gbtree",
            "tree_method": "hist",
            "n_estimators": trial.suggest_int("n_estimators", 1000, 3500, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 8),
            "max_leaves": trial.suggest_int("max_leaves", 32, 512),
            "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
            "gamma": trial.suggest_float("gamma", 0.01, 10.0, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.01, 10.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 5.0, log=True),
            "random_state": 42
        }
        
        scores = []
        for i in range(len(fold_datasets)):
            ranker = xgb.XGBRanker(**params)
            ranker.fit(
                fold_datasets[i]["X"],
                fold_datasets[i]["y"],
                group=fold_datasets[i]["groups"],
                verbose=False
            )
            
            reranker = XGBoostRerankerRecommender(
                URM_train=fold_datasets_val[i]["URM_train"],
                xgb_model=ranker,
                prediction_dataframe=fold_datasets_val[i]["full_df"]
            )
            
            # Evaluated on the held-out test fold
            from Evaluation.Evaluator import EvaluatorHoldout
            evaluator = EvaluatorHoldout(fold_datasets_val[i]["URM_test"], cutoff_list=[cutoff])
            res, _ = evaluator.evaluateRecommender(reranker)
            scores.append(res.loc[cutoff, "RECALL"] if "RECALL" in res.columns else res["RECALL"].values[0])
            
            del ranker, reranker, evaluator
            gc.collect()
            
        return float(np.mean(scores))
    
    return objective

print("5-Fold XGBRanker objective ready.")

## 5. Inspection of Real Historical Optimization Logs

Rather than re-running the 100 Optuna trials (which took ~14 hours of compute), we load and analyze the actual experimental log recorded in `results/xgb_optuna_trials.txt`.

In [ ]:
import re

log_path = "../results/xgb_optuna_trials.txt"
trials_data = []

if os.path.exists(log_path):
    with open(log_path, "r") as f:
        for line in f:
            match = re.search(r"Trial (\d+) finished with value: ([0-9\.]+) and parameters: (\{.*?\})", line)
            if match:
                trial_num = int(match.group(1))
                val = float(match.group(2))
                params = eval(match.group(3))
                params["trial"] = trial_num
                params["value"] = val
                trials_data.append(params)
    
    df_trials = pd.DataFrame(trials_data)
    print(f"Loaded {len(df_trials)} trials from real execution log!")
    print("Top 5 Trials by Validation Score:")
    display_cols = ["trial", "value", "n_estimators", "learning_rate", "max_depth", "max_leaves", "colsample_bytree"]
    print(df_trials.sort_values(by="value", ascending=False)[display_cols].head(5).to_string(index=False))
else:
    print(f"{log_path} not found.")

### Visualizing Optuna Convergence History
Let us plot the validation score trajectory across trials to show how the Bayesian optimization explored and converged to the optimal hyperparameter regime.

In [ ]:
if os.path.exists(log_path) and len(trials_data) > 0:
    plt.figure(figsize=(10, 5))
    plt.scatter(df_trials["trial"], df_trials["value"], alpha=0.6, c=df_trials["max_depth"], cmap="viridis", label="Trials (color=depth)")
    plt.plot(df_trials["trial"], df_trials["value"].cummax(), color="crimson", linewidth=2, label="Best So Far (CumMax)")
    plt.xlabel("Trial Number")
    plt.ylabel("Validation Score (MAP@20 / Recall@20)")
    plt.title("Stage 2 XGBRanker: Optuna Bayesian Optimization History")
    plt.colorbar(label="Max Depth")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("Plot skipped (no log data loaded).")

## 6. Real Feature Importance Analysis

We inspect the feature importance logs generated by the optimal XGBRanker model (`results/feature_importance_weights.txt`).

In [ ]:
fi_path = "../results/feature_importance_weights.txt"

if os.path.exists(fi_path):
    with open(fi_path, "r") as f:
        fi_lines = [line.strip() for line in f if line.strip()]
    print(f"Read {len(fi_lines)} lines of feature importance data.")
    print("\nFirst 15 lines of Feature Importance:")
    for line in fi_lines[:15]:
        print(" ", line)
else:
    print(f"{fi_path} not found.")

---
## Conclusion & Next Steps
- The 5-Fold Out-Of-Fold interaction validation protocol eliminated data leakage between the candidate generator and the reranker.
- Optuna tuning across 100 iterations on 5 folds identified the optimal tree hyperparameters (`max_depth=7`, `max_leaves=256-350`, `lossguide` grow policy, `colsample_bytree~0.55`).
- For final end-to-end training and test submission generation using these optimal hyperparameters, proceed to **[02_Final_XGBoost_Pipeline.ipynb](02_Final_XGBoost_Pipeline.ipynb)**.